# CNC Machine Tools — Exploratory Data Analysis (EDA)

**Business question:** *How can a buyer choose the most suitable CNC machine for a given budget and size requirement?*

**Source:** machinetoolindex.com, scraped and cleaned in this project (`01_scraping` → `03_cleaning`).
**Input file:** `07_data_outputs/machines_final.csv` (the final cleaned table, 29 columns, with outlier flags).

### Objectives
1. Understand the dataset: categories, manufacturers, available specifications.
2. Assess data quality: duplicates, missing values, outliers.
3. Analyze price distribution and compare categories.
4. Explore how price relates to size, weight, power and other specs.
5. State the limits of the data (estimated vs listed prices, partial scrape of the website).
6. Turn the findings into a simple budget-based machine finder.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9
plt.rcParams["figure.dpi"] = 110

## 2. Read data
The notebook lives in `06_notebook/`, so the data is one level up. The cell below tries a few locations so it also works if the notebook is moved.

In [ ]:
CANDIDATES = [
    Path("../07_data_outputs/machines_final.csv"),   # normal case: notebook inside 06_notebook
    Path("07_data_outputs/machines_final.csv"),      # notebook opened from the project root
    Path("machines_final.csv"),                      # file copied next to the notebook
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("machines_final.csv not found. Keep the project folder structure, or copy the file next to this notebook.")

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
print("Loaded:", DATA_PATH.resolve())
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
df.info()

## 3. Data cleaning (done before this notebook)
The data comes from a website, so the raw scrape is messy by nature: the same field has different names, numbers are text with mixed units, and prices are estimates. **The cleaning was done in `03_cleaning/` before this notebook starts**, so this notebook works on the already-cleaned table. This section documents what was done, proves it with before/after examples, and validates the result.

| Problem in the raw scrape | What we did | Script |
|---|---|---|
| ~5,200 different raw field names (same spec under many names, e.g. `X Axis Travel` and `X-Axis Travel`) | Merged duplicate columns, kept about 20 core columns, and the long table `specs_long` keeps every other spec | `clean_to_sqlite.py` |
| Specs stored as text with mixed units (`6,420 lb (2,915 kg)`) | Parsed into numbers with one unit (mm, kg, kW, rpm), ambiguous values left empty | `clean_to_sqlite.py` |
| Price stored as text (`Est. $200,000 - $275,000`) | Split into low / high / mid and `price_type` (estimate or listed) | `clean_to_sqlite.py`, `price_patch.py` |
| Broken characters (encoding) | Fixed at the source (UTF-8) | `scrape_all.py` |
| Manufacturer spelling variants, `Cnc` vs `CNC` | Merged to the most frequent spelling, standardized | `clean_and_visualize.py` |
| Impossible values (weight under 100 kg) | Set to missing | `clean_and_visualize.py` |
| Extreme values (huge machines) | **Flagged, not deleted**, because large machines are real | `clean_and_visualize.py` |
| Missing values | **Never guessed or filled**, left empty | all |

In [ ]:
# Before / after: raw scraped text vs the cleaned numeric columns (same machines, matched by URL)
import json
RAW_CANDIDATES = [Path("../02_raw_data/raw_machines.jsonl"), Path("02_raw_data/raw_machines.jsonl")]
RAW_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)

if RAW_PATH is None:
    print("raw_machines.jsonl not found, skipping the before/after example.")
else:
    with open(RAW_PATH, encoding="utf-8") as f:
        raw = [json.loads(line) for line in f]
    raw_by_url = {r["URL"]: r for r in raw}
    all_raw_fields = {k for r in raw for k in r}
    print(f"RAW:   {len(raw):,} records, {len(all_raw_fields):,} different field names, every value is text")
    print(f"CLEAN: {len(df):,} records, {df.shape[1]} columns (20 core + 9 helper/flag columns), specs as numbers")

    def first_raw(r, *keys):
        for k in keys:
            if k in r and r[k] not in (None, ""):
                return r[k]
        return None

    sample = df.dropna(subset=["x_travel_mm", "machine_weight_kg", "max_spindle_rpm", "price_usd"]).sample(6, random_state=1)
    rows = []
    for _, m in sample.iterrows():
        r = raw_by_url.get(m["URL"], {})
        rows.append({
            "Machine": m["Machine Name"],
            "price_text (raw)": m["price_text"],
            "price_usd": m["price_usd"],
            "RAW X travel": first_raw(r, "X Axis Travel", "X-Axis Travel"),
            "x_travel_mm": m["x_travel_mm"],
            "RAW weight": first_raw(r, "Machine Weight"),
            "machine_weight_kg": m["machine_weight_kg"],
            "RAW spindle speed": first_raw(r, "Max Spindle Speed"),
            "max_spindle_rpm": m["max_spindle_rpm"],
        })
    display(pd.DataFrame(rows))

    print("\nExample of duplicated columns that were merged into one:")
    for a, b in [("X Axis Travel", "X-Axis Travel"), ("Z Axis Travel", "Z-Axis Travel")]:
        print(f"  '{a}' ({sum(a in r for r in raw):,} records) + '{b}' ({sum(b in r for r in raw):,} records)  ->  one clean column")

In [ ]:
# The cleaning log written by clean_and_visualize.py: every issue found, how many rows, what we did
REPORT_CANDIDATES = [Path("../03_cleaning/cleaning_report.csv"), Path("03_cleaning/cleaning_report.csv"), Path("cleaning_report.csv")]
REPORT_PATH = next((p for p in REPORT_CANDIDATES if p.exists()), None)
if REPORT_PATH is None:
    print("cleaning_report.csv not found.")
else:
    report = pd.read_csv(REPORT_PATH, encoding="utf-8-sig")
    pd.set_option("display.max_colwidth", 90)
    display(report[~report["issue"].str.endswith("missing")])   # missing-value rows are shown in section 4

In [ ]:
# Validation: checks that the cleaned table really is clean
text_cols = df.select_dtypes("object").columns
garbled = sum(df[c].astype(str).str.contains("Ã|Â|â€", regex=True).sum() for c in ["Machine Name", "Manufacturer", "Category"])

checks = {
    "No duplicated rows":                      df.duplicated().sum() == 0,
    "URL is unique (1 row = 1 machine)":       df["URL"].is_unique,
    "No broken characters in names/categories": garbled == 0,
    "Category names consistent ('CNC' not 'Cnc')": not df["Category"].str.contains(r"\bCnc\b", regex=True).any(),
    "All numeric specs are positive":          all((df[c].dropna() > 0).all() for c in ["x_travel_mm","y_travel_mm","z_travel_mm","max_table_load_kg","machine_weight_kg","spindle_power_kw","max_spindle_rpm","tool_capacity","price_usd"]),
    "No machine under 100 kg":                 (df["machine_weight_kg"].dropna() >= 100).all(),
    "Price low <= mid <= high":                ((df["price_low_usd"] <= df["price_mid_usd"]) & (df["price_mid_usd"] <= df["price_high_usd"]) | df["price_mid_usd"].isna()).all(),
    "Every price has a type (estimate/listed)": df.loc[df["price_usd"].notna(), "price_type"].notna().all(),
}
display(pd.DataFrame({"check": checks.keys(), "passed": ["PASS" if v else "FAIL" for v in checks.values()]}))

**What the cleaning did NOT do, on purpose.**
- It did not fill missing values. A guessed spec or price would be a made-up number, so missing stays missing (see section 4).
- It did not delete extreme machines. Large gantry and boring machines are real, so they are flagged with `outlier_*` columns and the analysis below excludes outlier prices where it matters.
- It did not turn estimated prices into real prices. Every price chart must say "estimated" (see section 6.4).

## 4. Duplicates and missing values

In [ ]:
print("Duplicated rows:", df.duplicated().sum())
print("Duplicate machine URLs:", df["URL"].duplicated().sum())
print("Duplicate machine names:", df["Machine Name"].duplicated().sum(),
      "(flagged in column is_name_duplicate; same name can exist under different URLs)")

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing[missing.missing_count > 0])

## 5. Prepare numeric columns and the analysis subset
- `outlier_*` columns were created in the cleaning step (`clean_and_visualize.py`). A value of 1 marks an extreme value.
- Price analysis below uses machines with a **positive price**. Extreme prices are removed with `outlier_price_usd == 0`, and the number removed is printed.

In [ ]:
numeric_cols = [
    "price_usd", "price_low_usd", "price_high_usd", "price_mid_usd",
    "x_travel_mm", "y_travel_mm", "z_travel_mm",
    "max_table_load_kg", "machine_weight_kg",
    "spindle_power_kw", "max_spindle_rpm", "tool_capacity",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

priced_all = df[df["price_usd"] > 0]
priced = priced_all[priced_all["outlier_price_usd"] == 0].copy()

print("Total machine records:", len(df))
print("Records with a positive price:", len(priced_all), f"({len(priced_all)/len(df)*100:.1f}%)")
print("Price outliers excluded:", len(priced_all) - len(priced))
print("Records used for price analysis:", len(priced))

## 6. Dataset composition

#### 6.1 Machines by category

In [ ]:
category_counts = df["Category"].value_counts().sort_values()
fig, ax = plt.subplots()
category_counts.plot(kind="barh", ax=ax)
ax.set_title("Number of CNC Machines by Category")
ax.set_xlabel("Number of machine records")
ax.set_ylabel("Machine category")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.show()

#### 6.2 Top 10 manufacturers

In [ ]:
manufacturer_counts = df["Manufacturer"].value_counts().head(10).sort_values()
fig, ax = plt.subplots()
manufacturer_counts.plot(kind="barh", ax=ax)
ax.set_title("Top 10 Manufacturers by Number of Records")
ax.set_xlabel("Number of machine records")
ax.set_ylabel("Manufacturer")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.show()

#### 6.3 Price distribution

In [ ]:
fig, ax = plt.subplots()
ax.hist(priced["price_usd"], bins=30, edgecolor="black")
ax.set_title("Distribution of CNC Machine Prices")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("Number of machines")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()

print("Number of prices used:", len(priced))
print("Mean price:   ${:,.0f}".format(priced["price_usd"].mean()))
print("Median price: ${:,.0f}".format(priced["price_usd"].median()))

#### 6.4 Listed vs estimated prices

In [ ]:
price_type_counts = df["price_type"].fillna("No price").value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
price_type_counts.plot(kind="bar", ax=ax)
ax.set_title("Records by Price Type")
ax.set_xlabel("Price type")
ax.set_ylabel("Number of records")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()

display(priced.groupby("price_type")["price_usd"].agg(["count", "median", "mean"]).round(0))

#### 6.5 Median price by category

In [ ]:
price_category = (
    priced.groupby("Category")["price_usd"]
          .agg(["count", "median", "mean"])
          .query("count >= 20")
          .sort_values("median")
)
display(price_category.round(0))

fig, ax = plt.subplots()
price_category["median"].plot(kind="barh", ax=ax)
ax.set_title("Median Machine Price by Category")
ax.set_xlabel("Median price (USD)")
ax.set_ylabel("Machine category")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.show()

#### 6.6 Price spread by category

In [ ]:
box_labels = list(price_category.index)
box_data = [priced.loc[priced["Category"] == c, "price_usd"].values for c in box_labels]

fig, ax = plt.subplots(figsize=(12, 6))
try:
    ax.boxplot(box_data, tick_labels=box_labels, showfliers=False)   # Matplotlib >= 3.9
except TypeError:
    ax.boxplot(box_data, labels=box_labels, showfliers=False)        # older Matplotlib
ax.set_title("Price Distribution by CNC Machine Category")
ax.set_xlabel("Machine category")
ax.set_ylabel("Price (USD)")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()

## 7. Price versus specifications

#### 7.1 Price versus X-axis travel

Spearman correlation is reported next to Pearson because prices and sizes are heavily skewed. Spearman does not depend on extreme values.

In [ ]:
def relation(data, x, y, title, xlabel, ylabel, logx=False, logy=False):
    s = data.dropna(subset=[x, y])
    s = s[(s[x] > 0) & (s[y] > 0)]
    fig, ax = plt.subplots()
    ax.scatter(s[x], s[y], alpha=0.45)
    if logx: ax.set_xscale("log")
    if logy: ax.set_yscale("log")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.grid(True, which="both", alpha=0.25)
    plt.tight_layout(); plt.show()
    out = {"n": len(s)}
    if len(s) > 1:
        out["pearson"] = round(s[[x, y]].corr().iloc[0, 1], 3)
        out["spearman"] = round(s[[x, y]].corr(method="spearman").iloc[0, 1], 3)
        out["pearson_log"] = round(np.log(s[[x, y]]).corr().iloc[0, 1], 3)   # correlation of log values
    print(f"{x} vs {y}:", out)
    return out

r_x  = relation(priced, "x_travel_mm", "price_usd", "Machine Price vs. X-Axis Travel",
                "X-axis travel (mm)", "Price (USD, log scale)", logy=True)

#### 7.2 Machine weight versus price

In [ ]:
r_w = relation(priced, "machine_weight_kg", "price_usd", "Machine Weight vs. Price",
               "Machine weight (kg, log scale)", "Price (USD, log scale)", logx=True, logy=True)

#### 7.3 Other specifications versus price (summary)

In [ ]:
rows = []
for col in ["x_travel_mm", "machine_weight_kg", "spindle_power_kw", "tool_capacity", "max_spindle_rpm"]:
    s = priced.dropna(subset=[col]); s = s[s[col] > 0]
    rows.append({"spec": col, "records": len(s),
                 "spearman_with_price": round(s[[col, "price_usd"]].corr(method="spearman").iloc[0, 1], 3)})
spec_corr = pd.DataFrame(rows).sort_values("spearman_with_price", ascending=False)
display(spec_corr)

#### 7.4 X-axis travel versus Y-axis travel

In [ ]:
r_xy = relation(df, "x_travel_mm", "y_travel_mm", "X-Axis Travel vs. Y-Axis Travel",
                "X-axis travel (mm)", "Y-axis travel (mm)")
r_xy_log = relation(df, "x_travel_mm", "y_travel_mm", "X vs Y Travel (log-log)",
                    "X-axis travel (mm, log)", "Y-axis travel (mm, log)", logx=True, logy=True)

## 8. Data quality and overall relationships

#### 8.1 Missing values

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(11, 7))
missing_pct.head(15).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 15 Columns by Percentage of Missing Values")
ax.set_xlabel("Missing values (%)")
ax.set_ylabel("Column")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.show()

#### 8.2 Correlation heatmap

In [ ]:
corr_cols = ["price_usd", "x_travel_mm", "y_travel_mm", "z_travel_mm",
             "max_table_load_kg", "machine_weight_kg",
             "spindle_power_kw", "max_spindle_rpm", "tool_capacity"]
corr = df[corr_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right"); ax.set_yticklabels(corr.index)
ax.set_title("Spearman Correlation of Numeric Features")
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        v = corr.iloc[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, label="Spearman correlation")
plt.tight_layout(); plt.show()

#### 8.3 Spindle speed distribution

In [ ]:
rpm = df.loc[(df["max_spindle_rpm"] > 0) & (df["outlier_max_spindle_rpm"] == 0), "max_spindle_rpm"]
fig, ax = plt.subplots()
ax.hist(rpm, bins=30, edgecolor="black")
ax.set_title("Distribution of Maximum Spindle Speed (outliers excluded)")
ax.set_xlabel("Maximum spindle speed (RPM)")
ax.set_ylabel("Number of machines")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()
print("Records used:", len(rpm))
display(rpm.describe().round(0))

#### 8.4 Spec completeness by category

`spec_completeness` = how many of the 8 numeric specs are available for the machine (0 to 8).

In [ ]:
comp = df.groupby("Category")["spec_completeness"].mean().sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
comp.plot(kind="barh", ax=ax)
ax.set_title("Average Spec Completeness by Category (out of 8)")
ax.set_xlabel("Average number of specs available")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.show()

## 9. Budget-based machine finder
This answers the business question. Give a budget, a category and optionally a minimum X travel. The function returns the machines within the budget, with the most complete spec sheets first.
Prices are mostly estimates, so treat the result as a shortlist, not a quote.

In [ ]:
def find_machines(budget_usd, category=None, min_x_travel_mm=None, top=10):
    s = priced_all[priced_all["price_usd"] <= budget_usd]
    if category:
        s = s[s["Category"] == category]
    if min_x_travel_mm:
        s = s[s["x_travel_mm"] >= min_x_travel_mm]
    cols = ["Machine Name", "Manufacturer", "Category", "price_usd", "price_type",
            "x_travel_mm", "max_spindle_rpm", "spindle_power_kw", "spec_completeness"]
    return (s.sort_values(["spec_completeness", "price_usd"], ascending=[False, True])[cols]
              .head(top).reset_index(drop=True))

# Example: a 5-axis machining center up to $300,000 with at least 500 mm of X travel
display(find_machines(300_000, "5 Axis Machining Centers", min_x_travel_mm=500))

## 10. Summary for the report

In [ ]:
valid = priced_all["price_usd"]
print("=== DATASET SUMMARY (all positive prices, outliers included) ===")
print(f"Total machine records: {len(df):,}")
print(f"Categories: {df['Category'].nunique()}  |  Manufacturers: {df['Manufacturer'].nunique()}")
print(f"Records with a positive price: {len(valid):,} ({len(valid)/len(df)*100:.1f}%)")
print(f"Median price: ${valid.median():,.0f}   Mean price: ${valid.mean():,.0f}")
print(f"Min price: ${valid.min():,.0f}   Max price: ${valid.max():,.0f}")
display(df["Category"].value_counts().rename_axis("Category").to_frame("Record count"))

## Key insights
The numbers below are computed from the data above, so they stay correct if the data is refreshed.

In [ ]:
top_cat = df["Category"].value_counts()
pc = price_category
n_est = (priced_all["price_type"] == "estimate").sum()
n_list = (priced_all["price_type"] == "listed").sum()
med_by_type = priced.groupby("price_type")["price_usd"].median()
best_spec = spec_corr.iloc[0]
worst_spec = spec_corr.iloc[-1]

display(Markdown(f"""
**1. Dataset overview.** {len(df):,} machines, {df['Category'].nunique()} categories, {df['Manufacturer'].nunique()} manufacturers.
{top_cat.index[0]} is the largest category ({top_cat.iloc[0]:,}), followed by {top_cat.index[1]} ({top_cat.iloc[1]:,}).

**2. Scope limit.** Only {df['Category'].nunique()} of the 19 categories on the website were scraped, so conclusions apply to these categories only.

**3. Missing data.** Only {len(priced_all)/len(df)*100:.1f}% of machines have a price. The most-missing specs are table load, tool capacity and spindle power (see chart 8.1).

**4. Price distribution** (price outliers excluded). Median ${priced['price_usd'].median():,.0f} vs mean ${priced['price_usd'].mean():,.0f}. The mean is higher, so a few very expensive machines pull it up. The median is the better "typical price".

**5. Categories.** {pc.index[-1]} has the highest median price (${pc['median'].iloc[-1]:,.0f}) and {pc.index[0]} the lowest (${pc['median'].iloc[0]:,.0f}), among categories with at least 20 priced machines.

**6. Size and price.** X travel vs price: Spearman {r_x['spearman']} on {r_x['n']:,} machines. Weight vs price: Spearman {r_w['spearman']} on {r_w['n']:,} machines. Bigger and heavier machines cost more. This is an association, not proof of cause.

**7. Strongest and weakest price signal.** `{best_spec['spec']}` has the strongest rank correlation with price ({best_spec['spearman_with_price']}). `{worst_spec['spec']}` has the weakest ({worst_spec['spearman_with_price']}), so spindle speed alone says little about price.

**8. X and Y travel.** Pearson {r_xy['pearson']} on the raw values, but {r_xy['pearson_log']} on log values. The relationship is strong; very large machines hide it on the raw scale.

**9. Price reliability.** {n_est:,} of {n_est+n_list:,} prices are estimates and only {n_list:,} are listed. Median estimated price ${med_by_type.get('estimate', float('nan')):,.0f} vs listed ${med_by_type.get('listed', float('nan')):,.0f}. The two groups come from different machine mixes, so do not read this as estimates being inflated.
"""))

## Conclusion
Machine price follows machine size: bigger and heavier machines cost more, and the category matters a lot (lathes are the cheapest, boring machines and 5-axis machining centers the most expensive). A buyer should start from category and required travel, then use the budget filter in section 9 to build a shortlist.
The main limits are the low price coverage, the fact that nearly all prices are estimates, and that only part of the website was scraped. Improving these three points is the first step for future work.